In [ ]:
import pandas as pd
import os

In [ ]:
# зведення проміжних таблиць по групам

name1 = "..."
name2 = "..."

df1 = pd.read_excel(rf'...\{name1}.xlsx')
df2 = pd.read_excel(rf'...\{name2}.xlsx')

df1['from'] = f'{name1}'
df2['from'] = f'{name2}'

ids_1 = set(df1['id'])
ids_2 = set(df2['id'])

common_uuids = ids_1.intersection(ids_2)

df1_filtered = df1[df1['id'].isin(common_uuids)]
df2_filtered = df2[df2['id'].isin(common_uuids)]

merged_df = pd.concat([df1_filtered, df2_filtered], ignore_index=True)
sorted_df = merged_df.sort_values(by=['id', 'annotator'], ascending=[True, False])

sorted_df.to_excel(f'{name1}_{name2}.xlsx', index=False)

only_in_1 = ids_1 - ids_2
only_in_2 = ids_2 - ids_1

if only_in_1:
    print(f"Found only in {name1} (not in {name2}):")
    for uid in only_in_1:
        print(f" - {uid}")
else:
    print(f"The table {name1} has no unique ids (everything matches).")

print("-" * 30)

if only_in_2:
    print(f"Found only in {name2} (not in {name1}):")
    for uid in only_in_2:
        print(f" - {uid}")
else:
    print(f"The table {name2} has no unique ids (everything matches).")

In [ ]:
# зведення багатьох таблиць в одну


def concat_processed_files(file_names, base_path, output_name="multiple_to_one.xlsx"):
    dataframes = []

    for name in file_names:
        file_path = os.path.join(base_path, f"{name}.xlsx")

        if os.path.exists(file_path):
            df = pd.read_excel(file_path)
            dataframes.append(df)
            print(f" + Added: {name}.xlsx (rows: {len(df)})")
        else:
            print(f" - File {name}.xlsx not found")

    big_df = pd.concat(dataframes, ignore_index=True)

    if 'annotator' in big_df.columns:

        big_df['annotator'] = (
            big_df['annotator']
            .astype(str)
            .str.replace('Н', 'H')
            .str.replace('А', 'A')
            .str.replace('І', 'I')
            .str.replace('і', 'I')
        )

        big_df = big_df.sort_values(by=['id', 'annotator'], ascending=[True, False])

    output_path = os.path.join(base_path, output_name)
    big_df.to_excel(output_path, index=False)

    print("-" * 30)
    print(f"All tables joined and sorted.")
    print(f"Saved to {output_name}")
    print(f"Total rows: {len(big_df)}")

folder_path = r'...'

files_to_concat = [
    "S_A",
    "O_O",
    "B_M",
    "N_R",
    "Y_S",
    "D_O"
]

concat_processed_files(files_to_concat, folder_path, "One_Big_Table.xlsx")
